# 01 · Lakeflow pipeline run and data quality (executed)

**What this proves:** the serverless Lakeflow Spark Declarative Pipeline `kaapi_bricks_medallion` ran,
built 14 bronze → 14 silver → 7 gold tables, and every data-quality expectation was evaluated.
All numbers below come from the live workspace, the pipeline API and the pipeline **event log**.

In [1]:
from proof_helpers import *
print('catalog.schema:', FQ)

catalog.schema: fevm_cme_conde_catalog.kaapi_bricks


In [2]:
from datetime import datetime, timezone
pipe = [p for p in hq.pipelines.list_pipelines() if p.name.endswith("kaapi_bricks_medallion")][0]
print("pipeline:", pipe.name, "| id:", pipe.pipeline_id, "| state:", pipe.state)
spec = hq.pipelines.get(pipe.pipeline_id).spec
print("serverless:", spec.serverless, "| catalog.schema:", spec.catalog + "." + spec.schema)
print("\nlatest updates:")
for u in hq.pipelines.list_updates(pipe.pipeline_id, max_results=5).updates:
    print(f"  {u.update_id}  {u.state.value:<10} {datetime.fromtimestamp(u.creation_time/1000, timezone.utc):%Y-%m-%d %H:%M} UTC  cause={u.cause.value}")

pipeline: [dev kunal_gaurav] kaapi_bricks_medallion | id: 86ecf20f-fb31-43da-a626-db630e1dfa4b | state: PipelineState.IDLE


serverless: True | catalog.schema: fevm_cme_conde_catalog.kaapi_bricks

latest updates:


  3f04af09-d618-4ff6-ab57-a57983a116a9  COMPLETED  2026-10-23 17:59 UTC  cause=JOB_TASK
  4ae7fa51-c681-47b9-a776-f4672e792546  COMPLETED  2026-10-23 17:46 UTC  cause=API_CALL
  065d6a70-b298-4ccc-be9f-9c0433da45ea  COMPLETED  2026-09-27 09:25 UTC  cause=API_CALL
  8754e418-b66a-4cb7-ad14-b8b3143d5db9  FAILED     2026-09-27 09:24 UTC  cause=API_CALL
  e7f060ec-6df5-4972-8b05-f461036881a8  FAILED     2026-09-27 09:23 UTC  cause=API_CALL


## Update timeline from the event log (latest completed update)

In [3]:
latest = [u for u in hq.pipelines.list_updates(pipe.pipeline_id, max_results=10).updates if u.state.value == "COMPLETED"][0]
ev = df(f"""SELECT timestamp, event_type, message FROM event_log('{pipe.pipeline_id}')
           WHERE origin.update_id = '{latest.update_id}' AND event_type IN ('create_update','update_progress')
           ORDER BY timestamp""")
print("update:", latest.update_id)
print(ev.to_string(index=False))

update: 3f04af09-d618-4ff6-ab57-a57983a116a9
               timestamp      event_type                                 message
2026-10-23T17:59:28.109Z   create_update      Update 3f04af started by JOB_TASK.
2026-10-23T17:59:30.497Z update_progress Update 3f04af is WAITING_FOR_RESOURCES.
2026-10-23T17:59:41.243Z update_progress          Update 3f04af is INITIALIZING.
2026-10-23T17:59:59.983Z update_progress     Update 3f04af is SETTING_UP_TABLES.
2026-10-23T18:00:04.494Z update_progress               Update 3f04af is RUNNING.
2026-10-23T18:00:41.323Z update_progress             Update 3f04af is COMPLETED.


## Row counts per layer

In [4]:
tables = df(f"""SELECT table_name, table_type, comment FROM {CATALOG}.information_schema.tables
               WHERE table_schema = '{SCHEMA}' AND (table_name LIKE 'bronze_%' OR table_name LIKE 'gold_%'
               OR table_type = 'MATERIALIZED_VIEW') ORDER BY table_name""")
layer = lambda t: "bronze" if t.startswith("bronze_") else "gold" if t.startswith("gold_") else "silver"
tables["layer"] = tables.table_name.map(layer)
counts = []
for t in tables.table_name:
    counts.append(int(sql(f"SELECT COUNT(*) AS n FROM {FQ}.{t}")[0]["n"]))
tables["rows"] = counts
print(tables.groupby("layer").agg(tables=("table_name", "count"), rows=("rows", "sum")).to_string())
print()
print(tables[["layer", "table_name", "table_type", "rows"]].sort_values(["layer", "table_name"]).to_string(index=False))

        tables     rows
layer                  
bronze      14  1014636
gold         7    91521
silver      14  1014636

 layer                    table_name        table_type   rows
bronze              bronze_customers   STREAMING_TABLE  15000
bronze            bronze_ingredients   STREAMING_TABLE     22
bronze bronze_inventory_transactions   STREAMING_TABLE 171163
bronze    bronze_order_item_toppings   STREAMING_TABLE 261124
bronze            bronze_order_items   STREAMING_TABLE 325765
bronze                 bronze_orders   STREAMING_TABLE 200000
bronze          bronze_po_line_items   STREAMING_TABLE   3493
bronze               bronze_products   STREAMING_TABLE     28
bronze  bronze_promotion_redemptions   STREAMING_TABLE  35969
bronze             bronze_promotions   STREAMING_TABLE     15
bronze        bronze_purchase_orders   STREAMING_TABLE   2000
bronze                 bronze_stores   STREAMING_TABLE     37
bronze              bronze_suppliers   STREAMING_TABLE      8
bronze     

## Data-quality expectations (45 in silver), read from the event log

`DROP ROW` expectations remove invalid rows; the others warn. A constraint *passes* when it has 0 failed records.

In [5]:
dq = df(f"""
WITH e AS (
  SELECT origin.flow_name AS flow, explode(from_json(details:flow_progress:data_quality:expectations,
         'array<struct<name:string,dataset:string,passed_records:bigint,failed_records:bigint>>')) AS x
  FROM event_log('{pipe.pipeline_id}')
  WHERE details:flow_progress:data_quality:expectations IS NOT NULL
    AND origin.update_id = (SELECT origin.update_id FROM event_log('{pipe.pipeline_id}')
                            WHERE details:flow_progress:data_quality:expectations IS NOT NULL
                            ORDER BY timestamp DESC LIMIT 1))
SELECT x.dataset AS dataset, x.name AS expectation, SUM(x.passed_records) AS passed, SUM(x.failed_records) AS failed
FROM e GROUP BY 1, 2 ORDER BY 1, 2""")
dq[["passed", "failed"]] = dq[["passed", "failed"]].astype(int)
print(f"expectations evaluated: {len(dq)} | passed (0 failed rows): {(dq.failed == 0).sum()} | failed: {(dq.failed > 0).sum()}")
print(dq.assign(dataset=dq.dataset.str.split('.').str[-1]).to_string(index=False))

expectations evaluated: 45 | passed (0 failed rows): 45 | failed: 0
               dataset         expectation  passed  failed
             customers      has_home_store   15000       0
             customers   valid_customer_id   15000       0
             customers          valid_tier   15000       0
           ingredients        has_supplier      22       0
           ingredients   non_negative_cost      22       0
           ingredients valid_ingredient_id      22       0
inventory_transactions        non_neg_cost  171163       0
inventory_transactions        valid_txn_id  171163       0
inventory_transactions          valid_type  171163       0
   order_item_toppings       non_neg_price  261124       0
   order_item_toppings          valid_item  261124       0
   order_item_toppings        valid_oit_id  261124       0
   order_item_toppings       valid_topping  261124       0
           order_items      positive_price  325765       0
           order_items       valid_item_id  325